# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abc085455-byte/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


In [1]:
# --- Setup: make this notebook work whether it's opened locally or via the Colab badge ---
# Colab's working directory is /content, not the repo -- so relative paths like
# "data/raw/..." break unless we locate (or clone) the repo root first.
import os, pathlib, subprocess

REPO_URL = "https://github.com/abc085455-byte/flyrank-ml-internship.git"

def find_repo_root(start="."):
    p = pathlib.Path(start).resolve()
    for candidate in [p, *p.parents]:
        if (candidate / "skills" / "README.md").exists() and (candidate / "docs").exists():
            return candidate
    return None

repo_root = find_repo_root()
if repo_root is None:
    clone_dir = pathlib.Path("/content/flyrank-ml-internship")
    if not clone_dir.exists():
        subprocess.run(["git", "clone", REPO_URL, str(clone_dir)], check=True)
    repo_root = clone_dir

os.chdir(repo_root)
print("Working directory set to:", os.getcwd())


Working directory set to: /content/flyrank-ml-internship


In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining"] = (df["trend_direction"] == "down").astype(int)
print(f"rows: {len(df):,} | clients: {df['client_id'].nunique()} | overall decline rate: {df['is_declining'].mean():.3f}")


rows: 30,000 | clients: 32 | overall decline rate: 0.542


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

Every volume field here is heavily right-skewed — the mean sits far above the median, and the
p95/max gap is enormous. `impressions_90d` has a median of 731 but a max of 517,715 (a ~700x
spread); `clicks_90d`'s median is just 1 with a max of 4,178. That means a plain mean would be
dominated by a handful of huge pages, and it's why the model features in later notebooks use
`log1p()` on these fields rather than raw values. `content_age_days` and
`days_since_last_update`, by contrast, are much better-behaved (median close to mean, capped at a
few hundred days) — no log transform needed there.


In [3]:
for col in ["impressions_90d", "clicks_90d", "word_count", "content_age_days", "days_since_last_update"]:
    s = df[col].dropna()
    print(f"{col:24s} mean={s.mean():9.1f}  median={s.median():8.1f}  p95={s.quantile(0.95):9.1f}  max={s.max():9.1f}")


impressions_90d          mean=   5200.4  median=   731.0  p95=  22996.5  max= 517715.0
clicks_90d               mean=     16.1  median=     1.0  p95=     69.0  max=   4178.0
word_count               mean=   3107.8  median=  2877.0  p95=   6173.0  max=   9546.0
content_age_days         mean=    256.2  median=   236.0  p95=    487.0  max=    564.0
days_since_last_update   mean=     46.1  median=    20.0  p95=    104.0  max=    373.0


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

**Signal 1 — older content declines more (`content_age_days` vs. decline rate). Verdict: OPPOSITE.**
Decline rate falls as content gets older: `0-180d` = 62.7% (n=12,272), `181-365d` = 51.5%
(n=11,368), `366-730d` = 42.6% (n=6,360) — a clean, monotonic trend in the *opposite* direction
from the naive assumption that older content is more likely to be declining. A plausible reading:
older surviving pages have already settled into a stable audience or been through an earlier
refresh cycle, while newer pages are still finding (or losing) their footing. Either way, "old"
should not be used as a decline signal on its own here.

**Signal 2 — thinner content declines more (`word_count` vs. decline rate, visible pages only,
`impressions_90d >= 100`). Verdict: MIXED.** Not monotonic: `601-1200` words = 53.3% (n=152, a
thin cell), `1201-2000` = 78.2% (n=1,955, the highest rate of any bucket), `2000+` = 63.4%
(n=13,344) — versus an overall visible-page rate of 59.8%. The thinnest bucket is both small and
*below* average, which contradicts a simple "thin content decays more" story, and the highest
decline rate sits in the middle bucket, not either extreme. Word count alone isn't a clean
decline signal here, though it may still matter combined with other fields (as `w05_model.ipynb`
finds for the fitted model).

**Signal 3 — worse position means more decline (`avg_position` vs. decline rate, real position
data only, `impressions_90d >= 100`). Verdict: MIXED / partly OPPOSITE.** The very best positions
(`1-3`, n=555) actually show the *highest* decline rate (75.3%) of any bucket, while the *worst*
position bucket (`51+`, n=878) shows the *lowest* (31.8%) — the opposite of what "worse position
-> more decline" would predict at the tails. The middle bunches (`4-10`, `11-20`, `21-50`) all sit
close to the overall 59.8% rate. A plausible story: top-3 pages have the most to lose and the most
volatile competition for that top spot, while very-low-position pages may already be flat and
have little room left to decline further. Position alone doesn't cleanly predict decline
direction either.


In [4]:
df["age_bucket"] = pd.cut(df["content_age_days"], bins=[0, 180, 365, 730, 10000],
                           labels=["0-180", "181-365", "366-730", "731+"])
print("Signal 1 -- content_age_days vs decline rate")
print(df.groupby("age_bucket", observed=True)["is_declining"].agg(["mean", "count"]).round(3))

vis = df[df["impressions_90d"] >= 100].copy()
vis["wc_bucket"] = pd.cut(vis["word_count"], bins=[0, 600, 1200, 2000, 100000],
                           labels=["0-600", "601-1200", "1201-2000", "2000+"])
print("\nSignal 2 -- word_count vs decline rate (visible pages)")
print(vis.groupby("wc_bucket", observed=True)["is_declining"].agg(["mean", "count"]).round(3))
print("overall visible decline rate:", round(vis["is_declining"].mean(), 3))

posv = df[(df["avg_position"] > 0) & (df["impressions_90d"] >= 100)].copy()
posv["pos_bucket"] = pd.cut(posv["avg_position"], bins=[0, 3, 10, 20, 50, 1000],
                             labels=["1-3", "4-10", "11-20", "21-50", "51+"])
print("\nSignal 3 -- avg_position vs decline rate (visible, real position)")
print(posv.groupby("pos_bucket", observed=True)["is_declining"].agg(["mean", "count"]).round(3))
print("overall visible decline rate:", round(posv["is_declining"].mean(), 3))


Signal 1 -- content_age_days vs decline rate
             mean  count
age_bucket              
0-180       0.627  12272
181-365     0.515  11368
366-730     0.426   6360

Signal 2 -- word_count vs decline rate (visible pages)
            mean  count
wc_bucket              
601-1200   0.533    152
1201-2000  0.782   1955
2000+      0.634  13344
overall visible decline rate: 0.598

Signal 3 -- avg_position vs decline rate (visible, real position)
             mean  count
pos_bucket              
1-3         0.753    555
4-10        0.606   8660
11-20       0.626   5876
21-50       0.584   6037
51+         0.318    878
overall visible decline rate: 0.598


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

**Flag: `stale_visible_page`** — FlyRank's real refresh-flag logic assumes pages that haven't been
touched in a long time (`freshness_tier`) are more likely to be declining. **Verdict: MIXED.**
Decline rate is *not* monotonic in staleness: `0-30` days = 51.1% (n=20,480), `31-90` = 58.9%
(n=175, a thin cell), `91-180` = 61.1% (n=9,171), and the stalest bucket, `181+` = 47.1%
(n=174) — actually *below* the overall 54.2% rate. If staleness alone drove decline the way the
flag's story assumes, `181+` should be the highest bar, not the lowest. It isn't. On this data,
raw "hasn't been touched in N days" alone does not reliably predict "currently trending down" —
a real, clearly-stated negative result, not a shrug.


In [5]:
g = df.groupby("freshness_tier", observed=True)["is_declining"].agg(["mean", "count"]).round(3)
print(g.reindex(["0-30", "31-90", "91-180", "181+"]))
print("overall decline rate:", round(df["is_declining"].mean(), 3))


                 mean  count
freshness_tier              
0-30            0.511  20480
31-90           0.589    175
91-180          0.611   9171
181+            0.471    174
overall decline rate: 0.542


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

None of these individually-simple signals (age, word count, position, staleness) reliably predicts
decline on its own — each one is either flat, non-monotonic, or points the opposite direction from
the intuitive story. A content team should treat any single-field rule (including FlyRank's own
`stale_visible_page` staleness assumption) as a starting hypothesis to test on its own data, not a
fact to build a whole triage process on. This is exactly why the baseline rule in
`w04_baseline_score.ipynb` combines CTR *relative to position tier* rather than any one field in
isolation, and why the model in `w05_model.ipynb` is evaluated against that rule rather than
assumed to be better by default.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
